# Free AI video clips (Wan 2.1, on Kaggle's free GPU)

Makes short **portrait** (480x832) video clips from text prompts with **Wan 2.1 T2V 1.3B** (Alibaba, Apache-2.0 licence:
free, commercial use allowed). Each clip is about 5 seconds.

**Before you run:** on the right, *Settings -> Accelerator -> GPU T4 x2* and *Settings -> Internet -> On*
(Internet needs a phone-verified Kaggle account). Then click **Run All**. The clips appear in the *Output* panel
(and as `clips.zip`). One clip takes roughly 10-20 minutes on the free GPU; the first run also downloads the model (~5 min).


In [ ]:
# 1. Install (about 2 minutes)
!pip install -q -U "diffusers>=0.33" transformers accelerate ftfy imageio imageio-ffmpeg sentencepiece


In [ ]:
# 2. The prompts: one clip per line. Edit these for each video.
PROMPTS = [
    "Vertical close-up of a tired young man sitting on a night train. He slowly opens his mouth wide in a big, long yawn, "
    "eyes squeezing shut, then blinks and rubs his eyes. Dim warm carriage lights, city lights streaking past the window behind him, "
    "realistic, cinematic, shallow depth of field, natural skin texture.",
]
SEED = 42


In [ ]:
# 3. Load the model
import torch, gc
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import export_to_video

MODEL = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
print(torch.cuda.get_device_name(0))
# T4 has no fast bfloat16, so the big model runs in float16 and the picture decoder in float32
vae = AutoencoderKLWan.from_pretrained(MODEL, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(MODEL, vae=vae, torch_dtype=torch.float16)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=3.0)  # 3.0 suits 480p
pipe.enable_model_cpu_offload()  # keeps the 16 GB card from running out of memory


In [ ]:
# 4. Make the clips
import time, os
NEG = ("bright colours, overexposed, static, blurred details, subtitles, text, watermark, logo, worst quality, low quality, "
       "jpeg artifacts, ugly, deformed, extra fingers, poorly drawn hands, poorly drawn face, disfigured, fused fingers, "
       "still picture, messy background, many people in the background, walking backwards")
os.makedirs("/kaggle/working/clips", exist_ok=True)
for i, prompt in enumerate(PROMPTS, 1):
    t = time.time()
    frames = pipe(prompt=prompt, negative_prompt=NEG, height=832, width=480, num_frames=81,
                  num_inference_steps=30, guidance_scale=5.0,
                  generator=torch.Generator("cpu").manual_seed(SEED + i)).frames[0]
    out = f"/kaggle/working/clips/clip_{i:02d}.mp4"
    export_to_video(frames, out, fps=16)
    print(f"clip {i}: {out}  ({(time.time()-t)/60:.1f} min)")
    gc.collect(); torch.cuda.empty_cache()


In [ ]:
# 5. Bundle everything for download
!cd /kaggle/working && zip -qr clips.zip clips && ls -la clips
